# Upload One HPE COCO-Keypoints Image To Managed Datastore

This notebook uploads exactly one image plus one HPE annotation record in COCO keypoints format to the training API endpoint `POST /managed_datasets`.

It uses pycocotools to load one image and its COCO keypoints annotation.

The payload format matches the backend in `training/app/endpoints/dataset_endpoints.py`: 
- `file`: zip archive containing image file(s)
- `dataset_name`: dataset identifier
- `labels`: JSON list, one label per file
- `metadata`: JSON list, one metadata dict per file

Only COCO keypoints are supported in this notebook.

In [ ]:
import io
import json
import os
import zipfile
from datetime import datetime
from pathlib import Path

from pycocotools.coco import COCO
import requests

In [ ]:
# --- Configure these values ---
TRAINING_SERVER_URL = os.environ.get("TRAINING_SERVER_URL", "http://localhost:5253")
UPLOAD_URL = f"{TRAINING_SERVER_URL}/managed_datasets"

# COCO annotation JSON and image root
COCO_JSON_PATH = Path(r"C:\\Users\\mhjde\\source\\repos\\CHIMP\\hpe\\notebooks\\annotations.json")
COCO_IMAGES_ROOT = Path(r"C:\\Users\\mhjde\\source\\repos\\CHIMP\\hpe\\notebooks")

# Optional: set a specific image id from your COCO file. If None, first valid keypoint image is used.
COCO_IMAGE_ID = None

COCO_KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

if not COCO_JSON_PATH.exists():
    raise FileNotFoundError(f"COCO JSON not found: {COCO_JSON_PATH}")

coco = COCO(str(COCO_JSON_PATH))

if COCO_IMAGE_ID is not None:
    image_ids = [COCO_IMAGE_ID]
else:
    image_ids = list(coco.imgs.keys())

selected_img = None
selected_ann = None

for image_id in image_ids:
    ann_ids = coco.getAnnIds(imgIds=[image_id], iscrowd=False)
    anns = coco.loadAnns(ann_ids)
    for ann in anns:
        keypoints = ann.get("keypoints", [])
        if len(keypoints) == 51:
            selected_img = coco.loadImgs([image_id])[0]
            selected_ann = ann
            break
    if selected_img is not None:
        break

if selected_img is None or selected_ann is None:
    raise ValueError("No image with a valid 17-keypoint COCO annotation found")

IMAGE_PATH = COCO_IMAGES_ROOT / selected_img["file_name"]
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Image file not found from COCO metadata: {IMAGE_PATH}")

LABEL = "person"
ANNOTATION = {
    "format": "coco_keypoints",
    "image_id": selected_img.get("id"),
    "category_id": selected_ann.get("category_id", 1),
    "category_name": "person",
    "num_keypoints": selected_ann.get("num_keypoints", 17),
    "keypoint_names": COCO_KEYPOINT_NAMES,
    "keypoints": selected_ann.get("keypoints", []),
    "bbox": selected_ann.get("bbox", []),
    "area": selected_ann.get("area"),
    "iscrowd": selected_ann.get("iscrowd", 0),
    "source": "coco"
}

# Safe dataset id (no slashes or forbidden path characters)
DATASET_NAME = f"hpe_one_image_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print("Upload URL:", UPLOAD_URL)
print("Dataset:", DATASET_NAME)
print("COCO file:", COCO_JSON_PATH)
print("Selected image id:", selected_img.get("id"))
print("Selected annotation id:", selected_ann.get("id"))
print("Image path exists:", IMAGE_PATH.exists())

In [ ]:
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

if ANNOTATION.get("format") != "coco_keypoints":
    raise ValueError("Only 'coco_keypoints' format is supported in this notebook")

keypoints = ANNOTATION.get("keypoints", [])
if len(keypoints) != 51:
    raise ValueError(f"COCO keypoints must contain 51 values, got {len(keypoints)}")

if len(ANNOTATION.get("keypoint_names", [])) != 17:
    raise ValueError("COCO keypoint_names must contain 17 entries")

if len(ANNOTATION.get("bbox", [])) != 4:
    raise ValueError("COCO bbox must contain 4 values [x, y, w, h]")

# Build an in-memory zip with one file inside a dataset-like folder structure
zip_buffer = io.BytesIO()
archive_name = f"train/{LABEL}/{IMAGE_PATH.name}"

with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr(archive_name, IMAGE_PATH.read_bytes())

zip_buffer.seek(0)

labels = [LABEL]
metadata = [{
    "exp": "hpe",
    "filename": IMAGE_PATH.name,
    "annotation": ANNOTATION,
    "uploaded_at": datetime.utcnow().isoformat() + "Z"
}]

files = {
    "file": (f"{DATASET_NAME}.zip", zip_buffer.getvalue(), "application/zip"),
    "dataset_name": (None, DATASET_NAME),
    "labels": (None, json.dumps(labels)),
    "metadata": (None, json.dumps(metadata)),
}

response = requests.post(UPLOAD_URL, files=files, timeout=120)

print("Status:", response.status_code)
try:
    print("Response JSON:", json.dumps(response.json(), indent=2))
except Exception:
    print("Response text:", response.text)

## Notes
- Install dependency once in the kernel if needed: `pip install pycocotools`.
- If you get connection errors, verify `TRAINING_SERVER_URL` and that services are running.
- If you get a dataset name validation error, keep `DATASET_NAME` alphanumeric/underscore.
- The annotation is stored as metadata JSON and can be read later from the managed datastore database records.
- This notebook intentionally supports only one HPE annotation schema: COCO keypoints.